# Myanmar ASR with Whisper Small — combined dataset

Fine-tune `openai/whisper-small` on every eligible labeled training clip in `combined/myanmar_asr_all/`. Sources: local ASR, UCSY, FLEURS, and YODAS. VOA remains unlabeled and excluded. Validation and five test views stay held out; “full” means all eligible training data, not merging evaluation data into training.

Read [README.md](README.md) for setup, provenance, limitations, and citation. Run cells top to bottom. `QUICK_RUN = False` trains for three epochs; `True` checks 256 training and 64 validation clips for 20 steps. Audio is read and resampled on demand without a duplicate waveform cache. Metrics and prediction artifacts are tracked in the existing MLflow experiment. No training results are bundled with this notebook.


## 1. Install the notebook dependencies

First install PyTorch in this Jupyter kernel using the [official selector](https://pytorch.org/get-started/locally/) for your CPU/GPU. Restart the kernel after installing PyTorch. This cell installs the remaining packages; the notebook targets Transformers 4.57.1. Restart the kernel after this cell if Jupyter asks.


In [ ]:
from pathlib import Path
cwd = Path.cwd().resolve()
requirements = next(p / "notebook" / "requirements-training.txt" for p in (cwd, *cwd.parents)
                    if (p / "notebook" / "requirements-training.txt").is_file())
%pip install -q -r {requirements}


## 2. Set paths and check hardware

Start Jupyter inside this repository using the training environment in [README.md](README.md). Build the combined catalog first. Full mode uses every eligible training clip; quick mode selects 256 train and 64 validation clips deterministically.


In [ ]:
import csv
import os
import mlflow
import json
import random
import shutil
from pathlib import Path
from collections import Counter

import jiwer
import numpy as np
import torch
import transformers
from IPython.display import Audio, display
from tqdm.auto import tqdm
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          WhisperForConditionalGeneration, WhisperProcessor)

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "scripts" / "data" / "build_combined_catalog.py").is_file()
     and (p / "notebook" / "whisper-small").is_dir()),
    None,
)

print(ROOT)
assert ROOT is not None, f"Start Jupyter from inside myanmar-asr/: {cwd}"
import sys
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook"))
from training_common import (CatalogSpeechDataset, SpeechCollator, load_rows, filter_labels,
                             metric_function, decode_predictions, score_texts, save_predictions,
                             file_digest, tracked_run, MLflowMetricsCallback)
from scripts.data.build_combined_catalog import read_audio, text_key


In [ ]:
CATALOG_DIR = ROOT / "combined" / "myanmar_asr_all"
QUICK_RUN = False
MODEL_ID = "openai/whisper-small"
MODEL_REVISION = "main"  # Replace with a Hub commit SHA for a pinned experiment.
RUN_DIR = ROOT / "notebook" / "whisper-small" / ("run_combined_quick" if QUICK_RUN else "run_combined_full")
assert (CATALOG_DIR / "train_whisper.csv").is_file(), "Build the combined catalog first; see README.md"
for name in ("audio_data", "raw"):
    assert (CATALOG_DIR / name).is_dir(), f"Missing catalog audio directory/link: {name}; see README.md"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Catalog:", CATALOG_DIR)
print("Output:", RUN_DIR)

MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5050")
EXPERIMENT_NAME = "myanmar-asr-notebooks"
RUN_NAME = f"whisper-small-combined-{'quick' if QUICK_RUN else 'full'}"


In [ ]:
# Check the device cuda

device = "cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
print({"root": str(ROOT), "device": device, "torch": torch.__version__,
       "transformers": transformers.__version__, "ffmpeg": shutil.which("ffmpeg")})

if device == "cpu":
    print("CPU can check the data pipeline, but full Whisper training needs a GPU in practice.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.current_device())

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("Apple GPU (MPS) is available")

else:
    print("WARNING: Running on CPU")

In [ ]:
import shutil

print("ffmpeg:", shutil.which("ffmpeg"))
print("ffprobe:", shutil.which("ffprobe"))

## 3. Load curated splits and check leakage

All supervised duration views include labeled clips at most 30 seconds. Reject train/evaluation audio hashes and normalized transcript overlap. Record/video and inferred speaker overlap are reported in the catalog audit; this corpus does not guarantee speaker- or video-disjoint evaluation. Five named tests are evaluated separately after training.


In [ ]:
TEST_SPLIT_NAMES = ("test_asr", "test_user_study", "test_ucsy", "test_fleurs", "test_yodas_my")

def load_split(name):
    return load_rows(CATALOG_DIR / f"{name}_whisper.csv")

all_train = load_split("train")
all_validation = load_split("validation")
all_test_splits = {name: load_split(name) for name in TEST_SPLIT_NAMES}

def split_values(rows, key):
    return {text_key(row[key]) if key == "text" else row[key] for row in rows if row.get(key)}

for key in ("sha256", "text"):
    training_values = split_values(all_train, key)
    for name, rows in {"validation": all_validation, **all_test_splits}.items():
        assert not training_values & split_values(rows, key), f"{key} crosses train and {name}"
for name, rows in all_test_splits.items():
    assert not split_values(all_validation, "sha256") & split_values(rows, "sha256"), name

def summary(rows):
    return {"clips": len(rows), "hours": round(sum(float(r["duration_s"]) for r in rows) / 3600, 3),
            "sources": dict(sorted(Counter(r["source"] for r in rows).items()))}

print({name: summary(rows) for name, rows in
       {"train": all_train, "validation": all_validation, **all_test_splits}.items()})
rng = random.Random(20261030)
train_rows, validation_rows = all_train.copy(), all_validation.copy()
if QUICK_RUN:
    rng.shuffle(train_rows)
    rng.shuffle(validation_rows)
    train_rows = train_rows[:256]
    validation_rows = validation_rows[:64]
print("Selected:", summary(train_rows), summary(validation_rows))


## 4. Listen to two labels

These selected examples are a manual spot check, not an automatic proof that all labels are correct. Listen for text/audio mismatch, clipped speech, or a synthetic voice. If you find systematic problems, fix the manifest before trusting a model score.


In [ ]:
for row in train_rows[:2]:
    print(row["source"], row["sample_id"], row["text"])
    audio, rate = read_audio(row, CATALOG_DIR)
    display(Audio(audio.T, rate=rate))


## 5. Load Whisper and filter labels to its target length

The model downloads on first use. Myanmar transcription is selected explicitly. Rows whose target token sequence exceeds the model's limit are written to a review CSV, with counts printed here.


In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="myanmar", task="transcribe", revision=MODEL_REVISION)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model.generation_config.language = "myanmar"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False  # Required when using gradient checkpointing.
model.generation_config.use_cache = True
MAX_LABEL_TOKENS = model.config.max_target_positions

def filter_label_length(rows):
    return filter_labels(rows, processor, "whisper", model.config.decoder_start_token_id,
                         MAX_LABEL_TOKENS)

train_rows, rejected_train = filter_label_length(train_rows)
validation_rows, rejected_validation = filter_label_length(validation_rows)
assert train_rows and validation_rows
with (RUN_DIR / "long_labels.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["sample_id", "label_tokens", "reason", "text"])
    writer.writeheader()
    writer.writerows({key: row[key] for key in writer.fieldnames}
                     for row in rejected_train + rejected_validation)
print({"max_label_tokens": MAX_LABEL_TOKENS, "train": summary(train_rows),
       "validation": summary(validation_rows), "long_labels": len(rejected_train) + len(rejected_validation)})


## 6. Read audio without a waveform cache

The shared catalog reader supports local WAVs and audio embedded in saved Hugging Face datasets. The loader downmixes to mono and resamples to 16 kHz per access. It checks finite samples and duration rather than truncating a waveform against its full transcript. This saves disk space at the cost of repeated decoding/resampling. Hugging Face model downloads still use their normal cache.


In [ ]:
print("Reading original catalog audio on demand; no waveform cache is created.")


## 7. Create the training dataset and scoring function

The loader computes Whisper features when a clip is requested. The collator pads text labels and masks padding from the loss. The primary score is CER after removing whitespace, which reduces the effect of inconsistent Myanmar word spacing; raw CER is also reported. Both scores keep punctuation.


In [ ]:
collator = SpeechCollator("whisper", processor.tokenizer.pad_token_id,
                          model.config.decoder_start_token_id)
compute_metrics = metric_function(processor)
train_dataset = CatalogSpeechDataset(train_rows, CATALOG_DIR, processor.feature_extractor)
validation_dataset = CatalogSpeechDataset(validation_rows, CATALOG_DIR, processor.feature_extractor)
sample = train_dataset[0]
assert sample["input_features"].ndim == 2 and sample["labels"]
assert collator([sample])["labels"].shape[0] == 1
print("One sample loaded and collated successfully.")


## 8. Configure a short check or a full training run

The full run trains for three epochs using every eligible training clip and chooses the checkpoint with the lowest validation CER. Quick mode trains for 20 optimizer steps. CUDA uses mixed precision; other devices use full precision. Training uses one clip per GPU batch with 16 accumulation steps (effective batch size 16), and validation uses one clip per GPU batch to reduce peak memory. Resumable checkpoints are saved every 5 quick-run steps or 50 full-run steps, with up to three retained including the best. Combined-catalog runs have separate output folders from legacy mapping-only runs. If the display goes black during training, check the Windows System log for a GPU driver reset before retrying.


In [ ]:
interval = 20 if QUICK_RUN else 500
checkpoint_interval = 5 if QUICK_RUN else 50

class FrequentCheckpointCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % checkpoint_interval == 0:
            control.should_save = True
        return control

training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUN_DIR),
    logging_dir=str(RUN_DIR / "logs"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_steps=5 if QUICK_RUN else 100,
    num_train_epochs=3,
    max_steps=20 if QUICK_RUN else -1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=(device == "cuda"),
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=interval,
    save_strategy="steps",
    save_steps=interval,
    save_total_limit=3,
    logging_steps=5 if QUICK_RUN else 50,
    predict_with_generate=True,
    generation_max_length=MAX_LABEL_TOKENS,
    eval_accumulation_steps=4,
    dataloader_num_workers=0,  # Keeps notebook data loading reliable on Windows.
    load_best_model_at_end=True,
    metric_for_best_model="cer_no_space",
    greater_is_better=False,
    report_to=[],
    push_to_hub=False,
    seed=20261030,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[FrequentCheckpointCallback(), MLflowMetricsCallback()],
)
print("Output:", RUN_DIR, "| checkpoint every", checkpoint_interval, "steps")

configuration = {
    "family": "whisper",
    "tracking_uri": MLFLOW_URI,
    "experiment_name": EXPERIMENT_NAME,
    "model_id": MODEL_ID,
    "model_revision": model.config._commit_hash,
    "quick_run": QUICK_RUN,
    "manifest_sha256": {name: file_digest(CATALOG_DIR / f"{name}_whisper.csv")
                        for name in ("train", "validation", *TEST_SPLIT_NAMES)},
    "training_arguments": training_args.to_dict(),
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
    "selected": {"train": summary(train_rows), "validation": summary(validation_rows)},
    "rejected_labels": {"train": len(rejected_train), "validation": len(rejected_validation)},
}
config_path = RUN_DIR / "run_config.json"
if config_path.exists():
    if json.loads(config_path.read_text(encoding="utf-8")) != configuration:
        raise ValueError("Settings/data changed. Choose a new RUN_DIR for a new experiment.")
elif any(RUN_DIR.glob("checkpoint-*")) or (RUN_DIR / "baseline_metrics.json").exists():
    raise ValueError("Existing run has no configuration. Choose a new RUN_DIR.")
else:
    config_path.write_text(json.dumps(configuration, indent=2), encoding="utf-8")


## 9. Measure the pretrained baseline, then fine-tune

The baseline uses the exact same validation clips and scoring function as training. After a power cut, restart the kernel and rerun cells from the top; this cell reuses saved baseline metrics and resumes from the newest complete checkpoint. An interrupted checkpoint is skipped. After fine-tuning, the best checkpoint is saved with the processor. Compare the printed metrics before deciding whether the model improved.


In [ ]:
import zipfile

def latest_complete_checkpoint():
    checkpoints = sorted(
        (p for p in RUN_DIR.glob("checkpoint-*") if p.is_dir() and p.name[11:].isdigit()),
        key=lambda p: int(p.name[11:]), reverse=True,
    )
    required = ("trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth")
    for checkpoint in checkpoints:
        try:
            if not all((checkpoint / name).stat().st_size > 0 for name in required):
                raise ValueError("missing or empty resume state")
            if not any((checkpoint / name).stat().st_size > 0 for name in
                       ("model.safetensors", "pytorch_model.bin") if (checkpoint / name).is_file()):
                raise ValueError("missing or empty model weights")
            # A nonempty file can still be a truncated PyTorch ZIP archive.
            archives = [checkpoint / name for name in required[1:]]
            archives += [checkpoint / name for name in ("scaler.pt", "pytorch_model.bin")
                         if (checkpoint / name).is_file()]
            for path in archives:
                try:
                    with zipfile.ZipFile(path) as archive:
                        bad_member = archive.testzip()
                        if bad_member is not None:
                            raise ValueError(f"CRC check failed: {bad_member}")
                except (OSError, ValueError, zipfile.BadZipFile, EOFError, RuntimeError) as error:
                    raise ValueError(f"{path.name}: {error}") from error
            state = json.loads((checkpoint / "trainer_state.json").read_text(encoding="utf-8"))
            if state.get("global_step") != int(checkpoint.name[11:]):
                raise ValueError("trainer step does not match checkpoint folder")
            return checkpoint
        except (OSError, ValueError) as error:
            print(f"Skipping {checkpoint.name}: {error}")
    if checkpoints:
        raise RuntimeError(
            "No checkpoint passed the resume checks. Existing checkpoints were preserved. "
            "Restore a backup or use a new RUN_DIR to start over."
        )
    return None

with tracked_run(RUN_DIR, MLFLOW_URI, EXPERIMENT_NAME, RUN_NAME, configuration) as active:
    print("MLflow run:", f"{MLFLOW_URI}/#/experiments/{active.info.experiment_id}/runs/{active.info.run_id}")
    checkpoint = latest_complete_checkpoint()
    baseline_path = RUN_DIR / "baseline_metrics.json"
    try:
        baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
        print("Pretrained baseline (saved):", baseline)
    except (FileNotFoundError, json.JSONDecodeError):
        baseline = trainer.evaluate(metric_key_prefix="baseline")
        print("Pretrained baseline:", baseline)
        baseline_path.write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    print("Resuming from:" if checkpoint else "Starting new training in:", checkpoint or RUN_DIR)

    train_result = trainer.train(resume_from_checkpoint=str(checkpoint) if checkpoint else None)

    print("Training:", train_result.metrics)

    best_dir = RUN_DIR / "best_model"
    trainer.save_model(str(best_dir))
    processor.save_pretrained(str(best_dir))

    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    validation_metrics = trainer.evaluate(metric_key_prefix="validation")
    metrics = {"baseline": baseline, "train": train_result.metrics, "validation": validation_metrics,
               "best_checkpoint": trainer.state.best_model_checkpoint}
    (RUN_DIR / "training_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print("Validation after training:", validation_metrics)

    mlflow.log_metrics(baseline, step=0)
    mlflow.log_dict(metrics, "metrics/training_metrics.json")
    mlflow.log_dict(json.loads((CATALOG_DIR / "summary.json").read_text(encoding="utf-8")),
                    "dataset/catalog_summary.json")
    mlflow.log_artifact(str(RUN_DIR / "long_labels.csv"), artifact_path="dataset")
    mlflow.set_tag("best_checkpoint", str(trainer.state.best_model_checkpoint))
    mlflow.set_tag("local_best_model", str(best_dir))


## 10. Evaluate the five held-out test views

Full mode scores every test reference without the training label-length filter. Save separate predictions, WER, raw CER, and no-space CER as percentages. Report each source separately; these scores are not directly comparable to historical thesis results.


In [ ]:
# Decode held-out clips without teacher-forced labels or shortened references.
if not QUICK_RUN:
    with tracked_run(RUN_DIR, MLFLOW_URI, EXPERIMENT_NAME, RUN_NAME, configuration):
        test_metrics = {}
        for name, rows in all_test_splits.items():
            dataset = CatalogSpeechDataset(rows, CATALOG_DIR, processor.feature_extractor,
                                           include_labels=False)
            result = trainer.predict(dataset, metric_key_prefix=name)
            hypotheses = decode_predictions(processor, result.predictions)
            save_predictions(RUN_DIR / f"{name}_predictions.csv", rows, hypotheses)
            test_metrics[name] = {**summary(rows), **score_texts([row["text"] for row in rows], hypotheses)}
            (RUN_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
            print(name, test_metrics[name])
        mlflow.log_dict(test_metrics, "metrics/test_metrics.json")
        for name in all_test_splits:
            mlflow.log_artifact(str(RUN_DIR / f"{name}_predictions.csv"), artifact_path="predictions")
            mlflow.log_metrics({f"{name}_{key}": test_metrics[name][key]
                                for key in ("wer", "cer_raw", "cer_no_space")})

else:
    print("Quick run: held-out tests skipped.")
